# Modules

In [ ]:
#How to install modules from Jupyter Notebook (only works if pip or pip3 is installed, depending on Python version)
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install simpleaudio
!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install scipy
!{sys.executable} -m pip install soundfile
!{sys.executable} -m pip install primefac
!{sys.executable} -m pip install midiutil

In [ ]:
# Import modules after they have been installed. This piece of code allows you to actively use the installed 
# modules within

#The 'as' keyword dictates how you would like to call the module when utilizing the functions within it.

import time #This module allows you to calculate how much time passses when code runs. 

import numpy as np #This module allows you to utilize a plethora numerical tools. 

import simpleaudio as sa #This module allows you to actively play sound from your code.
import soundfile as sf #This module allows you to save your waveforms as a wav file for later use or in another digital-audio workstation (DAW).

from scipy import signal #This module contains waveforms that are not sin waves (found in numpy)

import matplotlib.pyplot as plt
from matplotlib import rc

from midiutil.MidiFile import MIDIFile # Module to export MIDI files!

In [ ]:
#This piece of code puts printed text in Computer Modern font and allows the use of LaTeX syntax.  
rc('text', usetex = True)

# Audio Functions

In [ ]:
#Convert audio file into n-bit
def bit_convert(audio, n = 16):
    #Ensure that highest value is in n-bit range
    if np.max(np.abs(audio)) > 0:
        audio *= (2**(n - 1) - 1)/np.max(np.abs(audio))
    

    # Convert to 16-bit data and return
    if n == 16:
        return audio.astype(np.int16)
    else:
        print("WIP")
        return 0
    
#Turn audio data in mono data
def mono (audio):
    length = len(audio[:,0])

    new_audio = np.zeros((length, 2))

    for i in range(0, length):
        mean = np.mean(audio[i])
        new_audio[:][i] += np.array([mean, mean])

    return new_audio

#Turn audio data into stereo data by spreading it across evenly. 
def splay(audio):
    num_channels = len(audio[0]);
    length = len(audio[:,0])

    new_audio = np.zeros((length, 2))

    for i in range(0, num_channels):
        new_audio[:, 0] += i*audio[:, i]/(num_channels - 1)
        new_audio[:, 1] += (num_channels - 1 - i)*audio[:, i]/(num_channels - 1)
    return new_audio


#Generate splayed audio of a waveform wf over num_channel channels 
#initially defined over a single channel
def gen_splay_audio(wf, f, t, num_chan, detune_amp = 0.02):
    
    if num_chan == 1: #mono
        notes = np.zeros((len(t), 2))
        detune = detune_amp*np.random.rand()
        φ = 2 * np.pi * np.random.rand()
        audio = wf(f * (1 + detune), t, φ)
        notes[:, 0] += audio
        notes[:, 1] += audio
        
        return notes
            
    elif num_chan >= 2: #higher
        
        notes = np.zeros((len(t), num_chan))
        for i in range(num_chan):
            detune = detune_amp*np.random.rand()
            ϕ = 2 * np.pi * np.random.rand()
            notes[:, i] += wf(f * (1 + detune), t, ϕ)
        return splay(notes)
    
    

def play(audio, f_s):
    # Start playback
    play_obj = sa.play_buffer(audio, 2, 2, f_s)

    # Wait for playback to finish before exiting
    play_obj.wait_done()


In [ ]:
# LECTURE 1 FUNCTIONS

#Fades in a waveform that lasts forover a time Δt with exponent p.
def fade_in(t, Δt, p = 1):
    if p < np.inf:
        return np.minimum(1, (t/Δt)**p) 
    else:
        return np.minimum(1, np.exp(t/Δt - 1))

#Fades out a waveform over a time Δt. Sustains at some value S. 
def fade_out(t, Δt, T, p = 1, S = 0):
    return np.maximum(S, np.minimum(1, ((T - t)/Δt)**p))

#a) An (A)ttack (H)old (D)ecay (S)ustain envelope. (R) is for release, which is not here in software!
def AHDS(t, A, H, D, S, p = 1, q = 1):
    return fade_in(t, A)*fade_out(t, D, A + H + D, S = S)

In [ ]:
# Good colors to use when plotting, remember vectors are 0-indexed!
color_inventory = (1/255.)*np.array([[0, 0, 0], [230, 159, 0], [86, 180, 233], [0, 158, 115], [240, 228, 66], [0, 114, 78], [213, 94, 0], [204, 121, 167]])


In [ ]:
# Last lecture we learned about Euclidean rhythms, developed by Godfried Toussaint in the following paper:
# https://cgm.cs.mcgill.ca/~godfried/publications/banff.pdf (See also https://erikdemaine.org/papers/DeepRhythms_CGTA/paper.pdf for even more information).

# In this series of exercises, you will develop a Euclidean rhythm generaterator step-by-step that will output a MIDI file you
# can utilize in Logic. 


# Solutions to Exercises 1-2

In [ ]:
#Returns the greatest common divisor (gcd) of two integers p and q, along with each step of the Euclidean algorithm. 
def gcd_steps(p, q):
    p_k, q_k = np.minimum(p, q), np.maximum(p, q) #Make sure p_k < q_k

    # We write each step as q_k = n_k x p_k + r_k, so that q_k mod p_k = r_k. We then store each p_k until r_k = 0.

    r_k = q_k % p_k
    p_vec = np.array([p_k])
    while r_k != 0:

        #Perform a step of the Euclidean Algorithm
        q_k = p_k
        p_k = r_k
        r_k = q_k % p_k
        
        p_vec = np.concatenate((p_vec, np.array([p_k]))) #Store the result of the current step.

    return p_vec #The last element of p_vec is the greatest common divisor of p and q!

#A simple implementation of the Euclidean rhythm algorithm by Godfried Toussaint.
def E(num_hits, num_notes):
    columns_vec = gcd_steps(num_hits, num_notes)
    columns_vec = columns_vec[columns_vec <= num_notes - num_hits]
    
    ones = np.full((num_hits, 1), np.array([1]))
    zeros = np.full((num_notes - num_hits, 1), np.array([0]))
    rhythm = np.concatenate((ones, zeros)).tolist()
    
    for p_k in columns_vec:
        column_lengths = np.array([len(x) for x in rhythm]) #Length of each column
        N_cols = len(np.where(column_lengths == column_lengths[-1])[0]) #Number of columns with the same length as the last column
        if (N_cols >= p_k and p_k > 1) or (N_cols > p_k and p_k == 1):
            for i in range(p_k):
                rhythm[i].append(rhythm[-1 - (p_k - 1) + i]) #Append entry 
                if len(rhythm[i]) > 1:
                    
                    rhythm[i] = np.hstack((rhythm[i])).tolist()
                    
            rhythm = rhythm[0:(len(rhythm) - p_k)]
            
    return np.hstack(rhythm)

def E_beat(num_hits, num_notes, repetitions, T):

    E_result = E(num_hits, num_notes)
    E_timing = np.where(E_result == 1)[0]/len(E_result)

    T_vec = np.linspace(0, T, repetitions + 1)
    ΔT = T_vec[1] - T_vec[0];

    return np.concatenate([T_vec[i] + ΔT*E_timing for i in range(repetitions)])
    

# Exercise 1: Euclid's Algorithm for the GCD

In [ ]:
#In this exercise you will create a function that executes the Euclidean algorithm. 

# A1) Define a function called gcd_steps which takes two arguments p and q. 
    # As a reminder, to define a function called cow that takes an argument pig, you would use the syntax
    # def cow(pig): 
        # <<cow's code>>

    #the 'def' keyword specifies that what comes after is a function, and the objects in parentheses are the arguments.

# A2) Define new variables p_k and q_k so that p_k is the smaller number and q_k is the bigger number.
# A3) Define a variable r_k that is the remainder of p_k into q_k.
# A4) Initialize an array called p_vec with the current value of p_k.
# A5) Your function now knows about three values, p_k, q_k and r_k. 
#     As a result, create a while loop that executes the remaining steps of the algorithm until the remainder reaches zero.
#     In each interance of the loop, redefine q_k to and p_k according to Euclid's algorithm and then recompute the remainder.
#     At the end of each interance of the loop, concatenate the newest value of p_k  to p_vec using p_vec = np.concatenate((p_vec, np.array([p_k])).

#A6) Return p_vec. The last element of p_vec is the gcd of p and q! The other elements are relevant for the next exercise generating Euclidean Rhythms. 

# Exercise #2: Euclidean Rhythms

In [ ]:
#In this exercise you will create a function that generates rhythms based on the Euclidean algorithm.

# B1) Define a function called E with two arguments num_hits and num_notes. 
#     num_notes will be the total length of the beat and num_hits the number of times you play a note in the beat.
#     We will represent a rhythm with 1's and 0's e.g. 100010001000100. In terms of the notation used in lecture
#     100010001000100 = [x . . . x . . . x . . .] (Make sure you remember how this rhythm sounds! Ask a mentor if you forget).  
# B2) Now define a new variable columns_vec to be the output of gcd_steps with inputs num_hits and num_notes. 
# B3) If the number of zeros is more than the number of 1's, the first step of Toussaint's algorithm is ill-defined (can you see why?)
    # Redefine columns_vec so that it only contains elements that are smaller than the number of zeros, i.e. less than num_notes - num_hits.
# B4) Create an array for the first step of Toussaint's algorithm. 
    # That is, create an array of arrays called rhythm with num_hits 1's followed by (num_notes - numhits) 0's. 
    # Each element of rhythm will be an array that we will concatenate later in accordance with Toussaint's algorithm.
    # Specifically, make sure each element is an array and not a float. 
    # Your result of this step should look something like [[1], [1], [1], [0], [0]]
# B5) Create a for loop that goes through each element p_k of columns_vec. 
# B6) Define a variable that computes the length array element in rhythm
# B7) Define a new variable N_cols that computes the number of arrays with the same length as the last element in rhythms.
#     Tip: For an array called cow, cow[-1] accesses the last element of the array in Python!
# B8) We will now go through each step of Toussaint's algorithm. Remember we only perform a step in the algorithm if there are
    # enough columns to move over. In code speak, create an if statement that executes only if N_col is larger than the number of 0's
    # you want to move (which is p_k from step B5).
# B9) Append the last p_k elements of rhythm onto the first p_k elements of rhythm and then delete the last p_k elements. 
# B10) Outside of the for loop of B5) return np.hstack(rhythm). Your should now have a Euclidean rhythm generator!
# B11) Check that your result for E(3, 8) is [10010010]. 


# Exercise 3: Using Logic!

In [ ]:
#Use the following function E_beat to utilize your Euclidean algorithm generator to output a MIDI file!

In [ ]:
def E_beat(num_hits, num_notes, repetitions, T):

    E_result = E(num_hits, num_notes)
    E_timing = np.where(E_result == 1)[0]/len(E_result)

    T_vec = np.linspace(0, T, repetitions + 1)
    ΔT = T_vec[1] - T_vec[0];

    return np.concatenate([T_vec[i] + ΔT*E_timing for i in range(repetitions)])

In [ ]:
# Set the sample rate and duration of the sound
f_s = 44100
T = 10.0  # Set the duration of the sample to any desired value (in seconds)
k = 7
n = 16
repetitions = 5
ts = np.concatenate((E_beat(k, n, repetitions, T), np.array([T])))
t_vec = np.empty(len(ts) - 1, dtype = object)
for i in range(len(t_vec)):
    t_vec[i] = np.arange(0, ts[i + 1] - ts[i], 1/f_s) # np.arange(start_time, end_time, bin_size)
num_chan = 2 #Number of channels.

In [ ]:
## This Cell Plays a Splayed Sine Wave
A = 1
f_0 = 40# Hz
ω_0 = 2 * np.pi * f_0 #Factor of 2π! To generate greek letters, use backslash, e.g. \omega, and then press tab after.
φ = 0.0

#Waveform
tone = lambda t: np.sin(ω_0 * t + φ) #The tone of interest

wf = lambda f, t, φ: fade_in(t, 0.1)*fade_out(t, 0.1, 0.1)*tone(t)  # Full (W)ave(f)orm

audio = np.empty(len(t_vec), dtype = object)
for i in range(len(t_vec)):
    audio[i] = gen_splay_audio(wf, f_0, t_vec[i], num_chan)

audio = bit_convert(np.concatenate(audio))
play(audio, f_s)


In [ ]:
# create your MIDI object
mf = MIDIFile(1)     # only 1 track

track = 0   # the only track
time = 0    # start at the beginning
bpm = 140

mf.addTrackName(track, time, "Sample Track")
mf.addTempo(track, time, bpm)

# add some notes
channel = 0
volume = 100

for i in range(len(ts) - 1):
    pitch = 60           # C4 (middle C)
    time = ts[i]            
    duration =  ts[i + 1] - ts[i]         # 1 beat long
    mf.addNote(track, channel, pitch, time, duration, volume)


# write it to disk
with open("euclidean_rhythm_output.mid", 'wb') as outf:
    mf.writeFile(outf)